In [ ]:
# ============================================================
# CELL 1 — Force Clean Install (do this, then RESTART RUNTIME)
# ============================================================

!pip install -q --upgrade --force-reinstall langchain-core
!pip install -q --upgrade langchain-community
!pip install -q --upgrade langchain-huggingface
!pip install -q faiss-cpu sentence-transformers datasets gradio huggingface_hub
!pip install -q --upgrade numpy
!pip install -q --upgrade sentence-transformers

print("✅ Done")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
✅ Done


In [ ]:
# ============================================================
# CELL 2 — Imports (no langchain-huggingface for LLM)
#
# We bypass langchain-huggingface's LLM wrapper entirely.
# Instead we call HuggingFace InferenceClient directly —
# more stable, fewer version conflicts.
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

# Dataset
from datasets import load_dataset

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Vector store
from langchain_community.vectorstores import FAISS

# ✅ Embeddings only from langchain_huggingface (stable)
from langchain_huggingface import HuggingFaceEmbeddings

# ✅ LLM via huggingface_hub directly (avoids the broken wrapper)
from huggingface_hub import InferenceClient

print("✅ All libraries imported successfully.")


✅ All libraries imported successfully.


In [ ]:
# ============================================================
# CELL 3 — Set Your HuggingFace API Token
#
# Get a FREE token at: https://huggingface.co/settings/tokens
# Click "New token" → Name it anything → Type: Read → Create
# Paste it below inside the quotes.
# ============================================================

HF_TOKEN = "place your huggingface token here"  # <-- REPLACE WITH YOUR TOKEN

os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

print("✅ HuggingFace token set.")

✅ HuggingFace token set.


In [ ]:
# ============================================================
# CELL 4 — Load the ChatDoctor Dataset
#
# Dataset: lavita/ChatDoctor-HealthCareMagic-100k
# Real doctor-patient consultations from HealthCareMagic
#
# We load 5000 rows to keep Colab fast.
# Increase NUM_SAMPLES up to 50000 for better answer quality
# (but building the index will take longer).
# ============================================================

print("⏳ Loading dataset from HuggingFace Hub...")

dataset = load_dataset(
    "lavita/ChatDoctor-HealthCareMagic-100k",
    split="train"
)

NUM_SAMPLES = 20000
dataset = dataset.select(range(NUM_SAMPLES))

print(f"✅ Loaded {len(dataset)} doctor-patient conversations.")
print("\n📋 Sample entry:")
print("PATIENT:", dataset[0]['input'][:200])
print("DOCTOR :", dataset[0]['output'][:200])


⏳ Loading dataset from HuggingFace Hub...


✅ Loaded 20000 doctor-patient conversations.

📋 Sample entry:
PATIENT: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out..
DOCTOR : Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom i


In [ ]:
# ============================================================
# CELL 5 — Convert Dataset Rows into LangChain Documents
#
# Each row becomes one Document containing both the patient
# question and the doctor's answer.
# We skip rows that are empty or have very short responses.
# ============================================================

print("⏳ Building document corpus...")

docs = []

for i, row in enumerate(dataset):
    # Skip empty or very short entries
    if not row.get('input') or not row.get('output'):
        continue
    if len(row['output'].strip()) < 30:
        continue

    # Combine patient question + doctor answer as one chunk
    content = (
        f"Patient: {row['input'].strip()}\n"
        f"Doctor: {row['output'].strip()}"
    )

    docs.append(
        Document(
            page_content=content,
            metadata={"source": "HealthCareMagic-100k", "index": i}
        )
    )

print(f"✅ Created {len(docs)} document chunks.")



⏳ Building document corpus...
✅ Created 19937 document chunks.


In [ ]:
# ============================================================
# CELL 6 — Load Embedding Model + Build FAISS Vector Store
#
# Embeddings convert text into vectors so we can do
# semantic similarity search (find the most relevant
# doctor answers for any patient question).
#
# all-MiniLM-L6-v2 is small, fast, and free — runs on CPU.
# FAISS stores the vectors locally for fast retrieval.
#
# ⏳ This cell takes 3–5 minutes for 5000 documents.
# ============================================================

print("⏳ Loading embedding model...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},          # swap to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded.")
print("⏳ Building FAISS vector store (this takes a few minutes)...")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

# Save to disk so you can reload it next session
vectorstore.save_local("doctor_vectorstore")

print("✅ Vector store built and saved.")
# ── Save to Google Drive so it persists across sessions ──────

from google.colab import drive
drive.mount('/content/drive')

# Save vectorstore to your Drive
vectorstore.save_local("/content/drive/MyDrive/doctor_vectorstore")
print("✅ Saved to Google Drive.")


⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.
⏳ Building FAISS vector store (this takes a few minutes)...
✅ Vector store built and saved.
Mounted at /content/drive
✅ Saved to Google Drive.


In [ ]:
# ── Cell 7: Reload from Google Drive ─────────────────────────

from google.colab import drive
drive.mount('/content/drive')

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.load_local(
    "/content/drive/MyDrive/doctor_vectorstore",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)
print("✅ Vector store reloaded from Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store reloaded from Google Drive.


In [ ]:
# ============================================================
# CELL 8 — Configure the Retriever
#
# MMR (Maximal Marginal Relevance) retrieves results that are
# both relevant AND diverse — avoids returning 5 near-identical
# chunks, which gives the LLM richer context to answer from.
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,        # return 5 final results
        "fetch_k": 20  # pick 5 diverse ones from 20 candidates
    }
)

print("✅ Retriever ready (MMR, top-5).")


✅ Retriever ready (MMR, top-5).


In [ ]:
# ============================================================
# CELL 9 — LLM Setup (Qwen2.5-7B)
# ============================================================

from huggingface_hub import InferenceClient

client = InferenceClient(model="Qwen/Qwen2.5-7B-Instruct", token=HF_TOKEN)

def call_llm(prompt_text: str) -> str:
    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are MedAssist, a professional medical information assistant. "
                    "CRITICAL RULES:\n"
                    "1. Respond in English only — never switch to any other language.\n"
                    "2. Be empathetic, clear, and concise.\n"
                    "3. Never assume the user has symptoms unless they stated them.\n"
                    "4. Always recommend consulting a qualified doctor for personal advice.\n"
                    "5. Do not repeat the disclaimer — it is added automatically."
                )
            },
            {"role": "user", "content": prompt_text}
        ],
        max_tokens=500,
        temperature=0.4,
    )
    return response.choices[0].message.content

print("✅ LLM ready (Qwen2.5-7B-Instruct).")

✅ LLM ready (Qwen2.5-7B-Instruct).


In [ ]:
# ============================================================
# CELL 10 — Fixed Prompt Template
# ============================================================

from langchain_core.prompts import PromptTemplate

medical_prompt_template = """You are MedAssist, a professional medical information assistant.

STRICT RULES:
- The doctor-patient examples below are REFERENCE MATERIAL ONLY — they are NOT about the current user.
- NEVER assume the current user has any symptoms, readings, or conditions mentioned in the reference examples.
- NEVER use specific numbers, readings, or personal details from the reference examples in your response.
- Answer ONLY what the user actually asked — do not invent context.
- If the user sends a greeting (hi, hello, hey), respond warmly and briefly — no medical content.
- If the user asks about anything unrelated to health or medicine (e.g. geography, politics, history, science, sports, population, countries, technology), respond ONLY with: "I'm only able to assist with medical and health-related questions. Please ask me about symptoms, conditions, medications, or general health advice." — do not answer the question.
- NEVER start your response with "I understand" — vary your opening phrases naturally.
- If the user asked for your well being (how are you, how are you doing), respond warmly, and ask how you can help them today.
- If the user message does not contain any greeting word, NEVER start your response with "Hello", "Hi", "Hey" or any greeting word — get straight to the answer.
- NEVER give the exact same response twice — always vary the wording, structure, and detail level.
- For general medical questions, explain the topic clearly and objectively.
- Always recommend consulting a qualified doctor for personal medical advice.
- Do not repeat the disclaimer — it is added automatically.

--- Reference Examples (NOT the current user's history) ---
{context}
-----------------------------------------------------------

Conversation so far:
{chat_history}

User: {question}
MedAssist:"""

PROMPT = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=medical_prompt_template
)

print("✅ Prompt updated.")

✅ Prompt updated.


In [ ]:
# ============================================================
# CELL 12 — Minimal response function (LLM handles everything)
# ============================================================

chat_history_store = []

DISCLAIMER = (
    "\n\n---\n"
    "⚠️ Medical Disclaimer: For general informational purposes only. "
    "Please consult a qualified healthcare professional for personalised guidance."
)


def ask_doctor(user_query: str, history: list) -> str:
    global chat_history_store

    if not user_query.strip():
        return "Please type a message."

    # Format last 4 turns of history
    history_text = ""
    for human, ai in chat_history_store[-4:]:
        history_text += f"User: {human}\nMedAssist: {ai}\n\n"

    # Retrieve doctor consultation context
    retrieved_docs = retriever.invoke(user_query)
    context = "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)

    # Fill prompt and call LLM
    filled_prompt = PROMPT.format(
        context=context,
        chat_history=history_text.strip(),
        question=user_query
    )

    try:
        answer = call_llm(filled_prompt)
    except Exception as e:
        return f"⚠️ LLM Error: {str(e)}"

    chat_history_store.append((user_query, answer))
    return answer + DISCLAIMER


print("✅ Response function ready.")

✅ Response function ready.


In [ ]:
# ============================================================
# CELL 13 — Quick Terminal Test
# ============================================================

# Clear history before testing — prevents leaking into Cell 14
chat_history_store = []

print("🧪 Running test query...\n")

test_query = "I have been having persistent headaches behind my eyes for the past week. What could be causing this?"
response = ask_doctor(test_query, [])

print(f"👤 PATIENT:\n{test_query}\n")
print(f"🩺 DOCTOR BOT:\n{response}")

# Clear history again after test — so Cell 14 starts fresh
chat_history_store = []
print("\n✅ History cleared — ready for Cell 14.")

🧪 Running test query...

👤 PATIENT:
I have been having persistent headaches behind my eyes for the past week. What could be causing this?

🩺 DOCTOR BOT:
Persistent headaches behind the eyes can be concerning. There are several potential causes, including tension headaches, migraines, sinus issues, or even eye strain. It's important to consider any other symptoms you might be experiencing, such as fever, vision changes, or nasal congestion.

I recommend scheduling an appointment with your healthcare provider to discuss these symptoms in detail. They may suggest further evaluation, such as a physical examination, blood tests, or imaging studies to determine the underlying cause.

In the meantime, try to maintain good hydration, ensure adequate rest, and avoid known triggers. Over-the-counter pain relievers like ibuprofen or acetaminophen can help manage the discomfort, but it's best to consult a doctor for personalized advice.

Please make an appointment with your doctor to get a proper 

In [ ]:
# ============================================================
# CELL 14 — Flask App + pyngrok
# Running chatbot locally in Colab for testing purposes
# ============================================================

!pip install -q flask flask-cors pyngrok

import os, threading, time
from pathlib import Path
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
from pyngrok import ngrok

HF_TOKEN    = "place your HuggingFace Token here"   # <-- your HuggingFace token
NGROK_TOKEN = "Place your ngrok token here"             # <-- your ngrok token

# ── Kill old processes ────────────────────────────────────────
os.system("fuser -k 5000/tcp 2>/dev/null")
ngrok.kill()
time.sleep(2)

# ── Clear any history from Cell 13 test ──────────────────────
chat_history_store = []

# ── HTML Template ─────────────────────────────────────────────
Path("templates").mkdir(exist_ok=True)

html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>MedAssist</title>
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
    :root {
      --primary: #d97706;        /* amber-orange — header, buttons, avatars */
      --primary-dark: #b45309;   /* darker orange for hover states */
      --bg: #f5f5f4;             /* warm off-white background */
      --border: #d6d3d1;         /* stone-300 border */
      --text: #1c1917;           /* stone-900 text */
      --muted: #78716c;          /* stone-500 muted text */
      --radius: 14px;
      --emergency: #dc2626;      /* red — kept for emergency messages */
    }
    body { font-family: 'Segoe UI', system-ui, sans-serif; background: var(--bg);
           height: 100vh; display: flex; flex-direction: column; overflow: hidden; }

    header { background: var(--primary); color: white; padding: 12px 20px;
             display: flex; align-items: center; justify-content: space-between;
             flex-shrink: 0; box-shadow: 0 2px 10px rgba(14,116,144,0.3); }
    .header-left { display: flex; align-items: center; gap: 12px; }
    .logo-icon   { font-size: 26px; }
    .header-title { font-size: 20px; font-weight: 700; }
    .header-sub   { font-size: 11px; opacity: 0.8; margin-top: 1px; }
    .header-stats { display: flex; gap: 12px; }
    .stat-badge { background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.2);
                  border-radius: 20px; padding: 4px 12px; font-size: 12px; color: white; }

    .disclaimer {
                  background: #fff7ed; border-bottom: 2px solid #fed7aa; color: #9a3412;
                  text-align: center; padding: 6px 16px; font-size: 12px;
                  font-weight: 500; flex-shrink: 0;
                   }

    .main { flex: 1; display: flex; flex-direction: column; max-width: 800px;
            width: 100%; margin: 0 auto; padding: 14px 14px 0; overflow: hidden; }

    #chat-window { flex: 1; background: white; border: 1px solid var(--border);
                   border-radius: var(--radius) var(--radius) 0 0; overflow-y: auto;
                   padding: 18px; display: flex; flex-direction: column;
                   gap: 16px; scroll-behavior: smooth; }

    .msg { display: flex; gap: 10px; max-width: 88%; animation: fadeUp 0.22s ease; }
    @keyframes fadeUp {
      from { opacity: 0; transform: translateY(8px); }
      to   { opacity: 1; transform: translateY(0); }
    }
    .msg.user { align-self: flex-end; flex-direction: row-reverse; }
    .msg.bot  { align-self: flex-start; }

    .avatar { width: 34px; height: 34px; border-radius: 50%;
              display: flex; align-items: center; justify-content: center;
              font-size: 17px; flex-shrink: 0; margin-top: 2px; }
    .msg.bot  .avatar { background: var(--primary); }
    .msg.user .avatar { background: #fed7aa; }   /* light orange */

    .bubble { padding: 12px 15px; border-radius: 16px; font-size: 13.5px;
              line-height: 1.65; white-space: pre-wrap; word-break: break-word; }
    .msg.bot  .bubble { background: #f8fdff; border: 1px solid var(--border);
                        border-top-left-radius: 4px; color: var(--text); }
    .msg.user .bubble { background: var(--primary); color: white; border-top-right-radius: 4px; }
    .bubble.emergency { background: #fff1f2; border: 2px solid var(--emergency);
                        border-top-left-radius: 4px; color: #7f1d1d; }

    .typing-dots { display: flex; gap: 4px; align-items: center; padding: 4px 0; }
    .typing-dots span { width: 7px; height: 7px; background: var(--primary);
                        border-radius: 50%; animation: bounce 1.2s infinite; opacity: 0.5; }
    .typing-dots span:nth-child(2) { animation-delay: 0.15s; }
    .typing-dots span:nth-child(3) { animation-delay: 0.3s; }
    @keyframes bounce {
      0%, 60%, 100% { transform: translateY(0); opacity: 0.4; }
      30%            { transform: translateY(-5px); opacity: 1; }
    }

    .input-area { background: white; border: 1px solid var(--border); border-top: none;
                  border-radius: 0 0 var(--radius) var(--radius); padding: 12px;
                  display: flex; gap: 8px; align-items: center; flex-shrink: 0;
                  margin-bottom: 14px; }
    #user-input { flex: 1; border: 1.5px solid var(--border); border-radius: 24px;
                  padding: 10px 16px; font-size: 13.5px; outline: none;
                  font-family: inherit; color: var(--text); transition: border-color 0.2s; }
    #user-input:focus  { border-color: var(--primary); }
    #send-btn { background: var(--primary); color: white; border: none; border-radius: 50%;
                width: 40px; height: 40px; font-size: 17px; cursor: pointer;
                display: flex; align-items: center; justify-content: center;
                flex-shrink: 0; transition: background 0.2s; }
    #send-btn:hover    { background: var(--primary-dark); }
    #send-btn:disabled { opacity: 0.5; cursor: not-allowed; }
    .reset-btn { background: transparent; border: 1.5px solid var(--border);
                 border-radius: 20px; padding: 8px 12px; font-size: 12px;
                 color: var(--muted); cursor: pointer; font-family: inherit;
                 white-space: nowrap; transition: all 0.15s; }
    .reset-btn:hover { border-color: var(--primary); color: var(--primary); }

    #chat-window::-webkit-scrollbar { width: 5px; }
    #chat-window::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
  </style>
</head>
<body>

  <header>
    <div class="header-left">
      <div class="logo-icon">&#x1FA7A;</div>
      <div>
        <div class="header-title">MedAssist</div>
        <div class="header-sub">AI Medical Assistant &mdash; Powered by RAG + ChatDoctor Dataset</div>
      </div>
    </div>
    <div class="header-stats">
      <div class="stat-badge">Queries: <span id="q-count">0</span></div>
      <div class="stat-badge" id="status-badge">&#9679; Ready</div>
    </div>
  </header>

  <div class="disclaimer">
    &#9888;&nbsp; For informational purposes only &mdash; always consult a qualified doctor for personal medical advice.
  </div>

  <div class="main">

    <div id="chat-window"></div>

    <div class="input-area">
      <button class="reset-btn" onclick="resetChat()">&#8634; New session</button>
      <input id="user-input" type="text" name="query"
             placeholder="Describe your symptoms or ask a health question..."
             autocomplete="off"/>
      <button id="send-btn" onclick="sendMessage()" title="Send">&#10148;</button>
    </div>
  </div>

  <script>
    const chatWindow  = document.getElementById('chat-window');
    const userInput   = document.getElementById('user-input');
    const qCountEl    = document.getElementById('q-count');
    const statusBadge = document.getElementById('status-badge');
    let   queryCount  = 0;

    // ── ngrok bypass header — added to every fetch call ────────
    const HEADERS = {
      'Content-Type': 'application/json',
      'ngrok-skip-browser-warning': '1'
    };

    const scrollToBottom = () => { chatWindow.scrollTop = chatWindow.scrollHeight; };

    function appendMessage(role, text) {
      const wrap = document.createElement('div');
      wrap.className = `msg ${role}`;

      const avatar = document.createElement('div');
      avatar.className = 'avatar';
      avatar.textContent = role === 'bot' ? '🩺' : '👤';

      const bubble = document.createElement('div');
      bubble.className  = `bubble${text.includes('🚨') ? ' emergency' : ''}`;
      bubble.textContent = text;

      wrap.append(avatar, bubble);
      chatWindow.appendChild(wrap);
      scrollToBottom();
    }

    function showTyping() {
      const wrap = document.createElement('div');
      wrap.className = 'msg bot'; wrap.id = 'typing';
      const avatar = document.createElement('div');
      avatar.className = 'avatar'; avatar.textContent = '🩺';
      const bubble = document.createElement('div');
      bubble.className = 'bubble';
      bubble.innerHTML = '<div class="typing-dots"><span></span><span></span><span></span></div>';
      wrap.append(avatar, bubble);
      chatWindow.appendChild(wrap);
      scrollToBottom();
    }

    function removeTyping() { document.getElementById('typing')?.remove(); }

    async function sendMessage() {
      const text = userInput.value.trim();
      if (!text) return;

      userInput.value = '';
      document.getElementById('send-btn').disabled = true;
      appendMessage('user', text);
      showTyping();
      statusBadge.innerHTML = '&#9679; Thinking...';

      try {
        const res  = await fetch('/chat', {
          method: 'POST',
          headers: HEADERS,
          body: JSON.stringify({ message: text })
        });
        const data = await res.json();
        removeTyping();
        appendMessage('bot', data.response || 'Sorry, I could not process that.');
        queryCount++;
        qCountEl.textContent  = queryCount;
        statusBadge.innerHTML = '&#9679; Ready';
      } catch (err) {
        removeTyping();
        appendMessage('bot', 'Connection error. Please try again.');
        statusBadge.innerHTML = '&#9679; Error';
      } finally {
        document.getElementById('send-btn').disabled = false;
        userInput.focus();
      }
    }

    async function resetChat() {
      try {
        const res  = await fetch('/reset', { method: 'POST', headers: HEADERS });
        const data = await res.json();
        chatWindow.innerHTML = '';
        queryCount           = 0;
        qCountEl.textContent = '0';
        appendMessage('bot', data.response);
        statusBadge.innerHTML = '&#9679; Ready';
      } catch {
        appendMessage('bot', 'Could not reset. Please refresh.');
      }
    }


    window.addEventListener('load', () => {
      appendMessage('bot',
        'Hello! I am MedAssist, your AI medical information assistant.\\n\\n' +
        'I am trained on real doctor-patient consultations and can help with ' +
        'questions about symptoms, conditions, medications, and general health advice.\\n\\n' +
        'How can I help you today?\\n\\n' +
        'Please note: I provide general information only. Always consult a real doctor for personal advice.'
      );
      userInput.focus();
    });

    userInput.addEventListener('keydown', (e) => {
      if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); sendMessage(); }
    });
  </script>
</body>
</html>"""

Path("templates/index.html").write_text(html, encoding="utf-8")
print(f"✅ templates/index.html written ({Path('templates/index.html').stat().st_size:,} bytes)")


# ── Flask App ─────────────────────────────────────────────────
flask_app = Flask(__name__, template_folder="templates")
CORS(flask_app)

query_counter = 0

@flask_app.after_request
def add_headers(response):
    response.headers['ngrok-skip-browser-warning'] = '1'
    response.headers['Access-Control-Allow-Origin'] = '*'
    return response

@flask_app.route("/")
def index():
    return render_template("index.html")

@flask_app.route("/chat", methods=["POST"])
def chat():
    global query_counter
    try:
        data    = request.get_json(force=True, silent=True) or {}
        message = data.get("message", "").strip()
        if not message:
            return jsonify({"response": "Please enter a message."})
        response       = ask_doctor(message, [])
        clean_response = response.split("---")[0].strip()
        query_counter += 1
        return jsonify({"response": clean_response, "queries": query_counter})
    except Exception as e:
        return jsonify({"response": f"Server error: {str(e)}"}), 500

@flask_app.route("/reset", methods=["POST", "GET"])
def reset():
    global query_counter, chat_history_store
    query_counter      = 0
    chat_history_store = []
    return jsonify({"response": "Session reset! Hello again — I am MedAssist.\n\nHow can I help you today?"})

@flask_app.route("/health")
def health():
    return jsonify({"status": "ok", "queries": query_counter})


# ── Launch Flask ──────────────────────────────────────────────
def run_flask():
    flask_app.run(host="0.0.0.0", port=5000, use_reloader=False, debug=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)
print("✅ Flask server started on port 5000.")

# ── Open ngrok tunnel ─────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)

print("=" * 55)
print("🏥  MedAssist is LIVE!")
print(f"🌐  Open this URL:\n    {public_url}")
print("=" * 55)
print(f"\nHealth check: {public_url}/health")

✅ templates/index.html written (12,114 bytes)
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask server started on port 5000.
🏥  MedAssist is LIVE!
🌐  Open this URL:
    NgrokTunnel: "https://stream-outscore-overview.ngrok-free.dev" -> "http://localhost:5000"

Health check: NgrokTunnel: "https://stream-outscore-overview.ngrok-free.dev" -> "http://localhost:5000"/health


In [ ]:
# ============================================================
# CELL 15 — Deploy to HuggingFace Spaces
# Synced with final notebook version
# ============================================================

!pip install -q huggingface_hub

# ── Set your details ──────────────────────────────────────────
HF_TOKEN    = "PLACE YOUR HUGGINFACE TOKEN HERE"  # <-- your HuggingFace token
HF_USERNAME = "PLACE YOUR HUGGINGFACE USERNAME HERE"                 # <-- your HuggingFace username
SPACE_NAME  = "NAME YOUR SPACE"                # <-- name for your Huggingface Space


# ════════════════════════════════════════════════════════════
# FILE 1 — app.py
# ════════════════════════════════════════════════════════════

app_code = '''
import os
import warnings
warnings.filterwarnings("ignore")

from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from huggingface_hub import InferenceClient

HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Dataset ───────────────────────────────────────────────────
print("⏳ Loading dataset...")
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
dataset = dataset.select(range(20000))

docs = []
for i, row in enumerate(dataset):
    if not row.get("input") or not row.get("output"):
        continue
    if len(row["output"].strip()) < 30:
        continue
    docs.append(Document(
        page_content=f"Patient: {row[\'input\'].strip()}\\nDoctor: {row[\'output\'].strip()}",
        metadata={"source": "HealthCareMagic-100k", "index": i}
    ))
print(f"✅ {len(docs)} documents loaded.")

# ── Embeddings + Vector store ─────────────────────────────────
print("⏳ Building vector store...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever   = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20}
)
print("✅ Vector store ready.")

# ── LLM ───────────────────────────────────────────────────────
client = InferenceClient(model="Qwen/Qwen2.5-7B-Instruct", token=HF_TOKEN)

def call_llm(prompt_text):
    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are MedAssist, a professional medical information assistant. "
                    "CRITICAL RULES:\\n"
                    "1. Respond in English only — never switch to any other language.\\n"
                    "2. Be empathetic, clear, and concise.\\n"
                    "3. Never assume the user has symptoms unless they stated them.\\n"
                    "4. Always recommend consulting a qualified doctor for personal advice.\\n"
                    "5. Do not repeat the disclaimer — it is added automatically."
                )
            },
            {"role": "user", "content": prompt_text}
        ],
        max_tokens=400,
        temperature=0.4,
    )
    return response.choices[0].message.content

# ── Prompt (matches Cell 10 exactly) ─────────────────────────
PROMPT = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template="""You are MedAssist, a professional medical information assistant.

STRICT RULES:
- The doctor-patient examples below are REFERENCE MATERIAL ONLY — they are NOT about the current user.
- NEVER assume the current user has any symptoms, readings, or conditions mentioned in the reference examples.
- NEVER use specific numbers, readings, or personal details from the reference examples in your response.
- Answer ONLY what the user actually asked — do not invent context.
- If the user sends a greeting (hi, hello, hey), respond warmly and briefly — no medical content.
- NEVER start your response with "I understand" — vary your opening phrases naturally.
- If the user asked for your well being (how are you, how are you doing), respond warmly, and ask how you can help them today.
- If the user message does not contain any greeting word, NEVER start your response with "Hello", "Hi", "Hey" or any greeting word — get straight to the answer.
- NEVER give the exact same response twice — always vary the wording, structure, and detail level.
- For general medical questions, explain the topic clearly and objectively.
- Always recommend consulting a qualified doctor for personal medical advice.
- Do not repeat the disclaimer — it is added automatically.

--- Reference Examples (NOT the current user\'s history) ---
{context}
-----------------------------------------------------------

Conversation so far:
{chat_history}

User: {question}
MedAssist:"""
)

# ── Response function (matches Cell 12 exactly) ───────────────
chat_history_store = []

DISCLAIMER = (
    "\\n\\n---\\n"
    "⚠️ Medical Disclaimer: For general informational purposes only. "
    "Please consult a qualified healthcare professional for personalised guidance."
)

def ask_doctor(user_query, history):
    global chat_history_store

    if not user_query.strip():
        return "Please type a message."

    # Format last 4 turns of history
    history_text = ""
    for human, ai in chat_history_store[-4:]:
        history_text += f"User: {human}\\nMedAssist: {ai}\\n\\n"

    # Retrieve doctor consultation context
    retrieved_docs = retriever.invoke(user_query)
    context = "\\n\\n---\\n\\n".join(doc.page_content for doc in retrieved_docs)

    # Fill prompt and call LLM
    filled_prompt = PROMPT.format(
        context=context,
        chat_history=history_text.strip(),
        question=user_query
    )

    try:
        answer = call_llm(filled_prompt)
    except Exception as e:
        return f"⚠️ LLM Error: {str(e)}"

    chat_history_store.append((user_query, answer))
    return answer + DISCLAIMER

# ── HTML (matches Cell 14 exactly) ───────────────────────────
HTML = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>MedAssist</title>
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
    :root {
      --primary: #d97706;        /* amber-orange — header, buttons, avatars */
      --primary-dark: #b45309;   /* darker orange for hover states */
      --bg: #f5f5f4;             /* warm off-white background */
      --border: #d6d3d1;         /* stone-300 border */
      --text: #1c1917;           /* stone-900 text */
      --muted: #78716c;          /* stone-500 muted text */
      --radius: 14px;
      --emergency: #dc2626;      /* red — kept for emergency messages */
    }
    body { font-family: \\'Segoe UI\\', system-ui, sans-serif; background: var(--bg);
           height: 100vh; display: flex; flex-direction: column; overflow: hidden; }
    header { background: var(--primary); color: white; padding: 12px 20px;
             display: flex; align-items: center; justify-content: space-between;
             flex-shrink: 0; box-shadow: 0 2px 10px rgba(14,116,144,0.3); }
    .header-left { display: flex; align-items: center; gap: 12px; }
    .logo-icon    { font-size: 26px; }
    .header-title { font-size: 20px; font-weight: 700; }
    .header-sub   { font-size: 11px; opacity: 0.8; margin-top: 1px; }
    .header-stats { display: flex; gap: 12px; }
    .stat-badge { background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.2);
                  border-radius: 20px; padding: 4px 12px; font-size: 12px; color: white; }
    .disclaimer { background: #fff7ed; border-bottom: 2px solid #fed7aa; color: #9a3412;
                  text-align: center; padding: 6px 16px; font-size: 12px;
                  font-weight: 500; flex-shrink: 0;
                   }
    .main { flex: 1; display: flex; flex-direction: column; max-width: 800px;
            width: 100%; margin: 0 auto; padding: 14px 14px 0; overflow: hidden; }

    #chat-window { flex: 1; background: white; border: 1px solid var(--border);
                   border-radius: var(--radius) var(--radius) 0 0; overflow-y: auto;
                   padding: 18px; display: flex; flex-direction: column;
                   gap: 16px; scroll-behavior: smooth; }
    .msg { display: flex; gap: 10px; max-width: 88%; animation: fadeUp 0.22s ease; }
    @keyframes fadeUp { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }
    .msg.user { align-self: flex-end; flex-direction: row-reverse; }
    .msg.bot  { align-self: flex-start; }
    .avatar { width: 34px; height: 34px; border-radius: 50%; display: flex; align-items: center;
              justify-content: center; font-size: 17px; flex-shrink: 0; margin-top: 2px; }
    .msg.bot  .avatar { background: var(--primary); }
    .msg.user .avatar { background: #fed7aa; }   /* light orange */
    .bubble { padding: 12px 15px; border-radius: 16px; font-size: 13.5px;
              line-height: 1.65; white-space: pre-wrap; word-break: break-word; }
    .msg.bot  .bubble { background: #f8fdff; border: 1px solid var(--border);
                        border-top-left-radius: 4px; color: var(--text); }
    .msg.user .bubble { background: var(--primary); color: white; border-top-right-radius: 4px; }
    .bubble.emergency { background: #fff1f2; border: 2px solid var(--emergency);
                        border-top-left-radius: 4px; color: #7f1d1d; }
    .typing-dots { display: flex; gap: 4px; align-items: center; padding: 4px 0; }
    .typing-dots span { width: 7px; height: 7px; background: var(--primary);
                        border-radius: 50%; animation: bounce 1.2s infinite; opacity: 0.5; }
    .typing-dots span:nth-child(2) { animation-delay: 0.15s; }
    .typing-dots span:nth-child(3) { animation-delay: 0.3s; }
    @keyframes bounce { 0%, 60%, 100% { transform: translateY(0); opacity: 0.4; }
                        30% { transform: translateY(-5px); opacity: 1; } }
    .input-area { background: white; border: 1px solid var(--border); border-top: none;
                  border-radius: 0 0 var(--radius) var(--radius); padding: 12px;
                  display: flex; gap: 8px; align-items: center; flex-shrink: 0; margin-bottom: 14px; }
    #user-input { flex: 1; border: 1.5px solid var(--border); border-radius: 24px;
                  padding: 10px 16px; font-size: 13.5px; outline: none;
                  font-family: inherit; color: var(--text); transition: border-color 0.2s; }
    #user-input:focus { border-color: var(--primary); }
    #send-btn { background: var(--primary); color: white; border: none; border-radius: 50%;
                width: 40px; height: 40px; font-size: 17px; cursor: pointer;
                display: flex; align-items: center; justify-content: center;
                flex-shrink: 0; transition: background 0.2s; }
    #send-btn:hover    { background: var(--primary-dark); }
    #send-btn:disabled { opacity: 0.5; cursor: not-allowed; }
    .reset-btn { background: transparent; border: 1.5px solid var(--border); border-radius: 20px;
                 padding: 8px 12px; font-size: 12px; color: var(--muted); cursor: pointer;
                 font-family: inherit; white-space: nowrap; transition: all 0.15s; }
    .reset-btn:hover { border-color: var(--primary); color: var(--primary); }
    #chat-window::-webkit-scrollbar { width: 5px; }
    #chat-window::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
  </style>
</head>
<body>
  <header>
    <div class="header-left">
      <div class="logo-icon">&#x1FA7A;</div>
      <div>
        <div class="header-title">MedAssist</div>
        <div class="header-sub">AI Medical Assistant &mdash; Powered by RAG + ChatDoctor Dataset</div>
      </div>
    </div>
    <div class="header-stats">
      <div class="stat-badge">Queries: <span id="q-count">0</span></div>
      <div class="stat-badge" id="status-badge">&#9679; Ready</div>
    </div>
  </header>
  <div class="disclaimer">
    &#9888;&nbsp; For informational purposes only &mdash; always consult a qualified doctor for personal medical advice.
  </div>
  <div class="main">
    <div id="chat-window"></div>
    <div class="input-area">
      <button class="reset-btn" onclick="resetChat()">&#8634; New session</button>
      <input id="user-input" type="text" name="query"
             placeholder="Describe your symptoms or ask a health question..."
             autocomplete="off"/>
      <button id="send-btn" onclick="sendMessage()" title="Send">&#10148;</button>
    </div>
  </div>
  <script>
    const chatWindow  = document.getElementById(\\'chat-window\\');
    const userInput   = document.getElementById(\\'user-input\\');
    const qCountEl    = document.getElementById(\\'q-count\\');
    const statusBadge = document.getElementById(\\'status-badge\\');
    let   queryCount  = 0;
    const scrollToBottom = () => { chatWindow.scrollTop = chatWindow.scrollHeight; };

    function appendMessage(role, text) {
      const wrap = document.createElement(\\'div\\');
      wrap.className = `msg ${role}`;
      const avatar = document.createElement(\\'div\\');
      avatar.className = \\'avatar\\';
      avatar.textContent = role === \\'bot\\' ? \\'🩺\\' : \\'👤\\';
      const bubble = document.createElement(\\'div\\');
      bubble.className  = `bubble${text.includes(\\'🚨\\') ? \\' emergency\\' : \\'\\'}`;
      bubble.textContent = text;
      wrap.append(avatar, bubble);
      chatWindow.appendChild(wrap);
      scrollToBottom();
    }

    function showTyping() {
      const wrap = document.createElement(\\'div\\');
      wrap.className = \\'msg bot\\'; wrap.id = \\'typing\\';
      const avatar = document.createElement(\\'div\\');
      avatar.className = \\'avatar\\'; avatar.textContent = \\'🩺\\';
      const bubble = document.createElement(\\'div\\');
      bubble.className = \\'bubble\\';
      bubble.innerHTML = \\'<div class="typing-dots"><span></span><span></span><span></span></div>\\';
      wrap.append(avatar, bubble);
      chatWindow.appendChild(wrap);
      scrollToBottom();
    }

    function removeTyping() { document.getElementById(\\'typing\\')?.remove(); }

    async function sendMessage() {
      const text = userInput.value.trim();
      if (!text) return;
      userInput.value = \\'\\';
      document.getElementById(\\'send-btn\\').disabled = true;
      appendMessage(\\'user\\', text);
      showTyping();
      statusBadge.innerHTML = \\'&#9679; Thinking...\\';
      try {
        const res  = await fetch(\\'/chat\\', {
          method: \\'POST\\',
          headers: { \\'Content-Type\\': \\'application/json\\' },
          body: JSON.stringify({ message: text })
        });
        const data = await res.json();
        removeTyping();
        appendMessage(\\'bot\\', data.response || \\'Sorry, I could not process that.\\');
        queryCount++;
        qCountEl.textContent  = queryCount;
        statusBadge.innerHTML = \\'&#9679; Ready\\';
      } catch (err) {
        removeTyping();
        appendMessage(\\'bot\\', \\'Connection error. Please try again.\\');
        statusBadge.innerHTML = \\'&#9679; Error\\';
      } finally {
        document.getElementById(\\'send-btn\\').disabled = false;
        userInput.focus();
      }
    }

    async function resetChat() {
      try {
        const res  = await fetch(\\'/reset\\', { method: \\'POST\\' });
        const data = await res.json();
        chatWindow.innerHTML = \\'\\';
        queryCount = 0; qCountEl.textContent = \\'0\\';
        appendMessage(\\'bot\\', data.response);
        statusBadge.innerHTML = \\'&#9679; Ready\\';
      } catch { appendMessage(\\'bot\\', \\'Could not reset. Please refresh.\\'); }
    }


    window.addEventListener(\\'load\\', () => {
      appendMessage(\\'bot\\',
        \\'Hello! I am MedAssist, your AI medical information assistant.\\\\n\\\\n\\' +
        \\'I am trained on real doctor-patient consultations and can help with \\' +
        \\'questions about symptoms, conditions, medications, and general health advice.\\\\n\\\\n\\' +
        \\'How can I help you today?\\\\n\\\\n\\' +
        \\'Please note: I provide general information only. Always consult a real doctor for personal advice.\\'
      );
      userInput.focus();
    });

    userInput.addEventListener(\\'keydown\\', (e) => {
      if (e.key === \\'Enter\\' && !e.shiftKey) { e.preventDefault(); sendMessage(); }
    });
  </script>
</body>
</html>"""

# ── Flask ─────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)
query_counter = 0

@app.route("/")
def index():
    return render_template_string(HTML)

@app.route("/chat", methods=["POST"])
def chat():
    global query_counter
    try:
        data    = request.get_json(force=True, silent=True) or {}
        message = data.get("message", "").strip()
        if not message:
            return jsonify({"response": "Please enter a message."})
        response       = ask_doctor(message, [])
        clean_response = response.split("---")[0].strip()
        query_counter += 1
        return jsonify({"response": clean_response, "queries": query_counter})
    except Exception as e:
        return jsonify({"response": f"Error: {str(e)}"}), 500

@app.route("/reset", methods=["POST", "GET"])
def reset():
    global query_counter, chat_history_store
    query_counter = 0
    chat_history_store = []
    return jsonify({"response": "Session reset! Hello again — I am MedAssist.\\n\\nHow can I help you today?"})

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=7860)
'''

with open("app.py", "w") as f:
    f.write(app_code)
print("✅ app.py written")


# ════════════════════════════════════════════════════════════
# FILE 2 — requirements.txt
# ════════════════════════════════════════════════════════════

requirements = """flask
flask-cors
langchain-core
langchain-community
langchain-huggingface
faiss-cpu
sentence-transformers
datasets
huggingface_hub
numpy
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("✅ requirements.txt written")


# ════════════════════════════════════════════════════════════
# FILE 3 — Dockerfile
# ════════════════════════════════════════════════════════════

dockerfile = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 7860

CMD ["python", "app.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)
print("✅ Dockerfile written")


# ════════════════════════════════════════════════════════════
# FILE 4 — README.md
# ════════════════════════════════════════════════════════════

readme = f"""---
title: MedAssist
emoji: 🩺
colorFrom: indigo
colorTo: blue
sdk: docker
app_port: 7860
pinned: false
---

# 🩺 MedAssist — AI Medical Information Assistant

Built with LangChain, FAISS RAG, and the ChatDoctor-HealthCareMagic dataset.

> ⚠️ For informational purposes only — always consult a qualified doctor.
"""

with open("README.md", "w") as f:
    f.write(readme)
print("✅ README.md written")


# ════════════════════════════════════════════════════════════
# PUSH TO HUGGINGFACE SPACES
# ════════════════════════════════════════════════════════════

from huggingface_hub import HfApi

api     = HfApi()
repo_id = f"{HF_USERNAME}/{SPACE_NAME}"

print(f"\n⏳ Creating Space: {repo_id} ...")
api.create_repo(
    repo_id=repo_id,
    token=HF_TOKEN,
    repo_type="space",
    space_sdk="docker",
    exist_ok=True
)
print("✅ Space created.")

print("⏳ Adding HF_TOKEN secret...")
api.add_space_secret(
    repo_id=repo_id,
    key="HF_TOKEN",
    value=HF_TOKEN,
    token=HF_TOKEN
)
print("✅ Secret added.")

for filename in ["app.py", "requirements.txt", "Dockerfile", "README.md"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="space",
        token=HF_TOKEN
    )
    print(f"✅ Uploaded {filename}")

print(f"""
{'='*55}
🚀 MedAssist deployed to HuggingFace Spaces!
🌐 URL: https://huggingface.co/spaces/{HF_USERNAME}/{SPACE_NAME}
{'='*55}

⏳ Build takes 3-5 mins. First load takes ~5 mins
   while it downloads the dataset and builds the
   FAISS index. After that it responds instantly.
""")

✅ app.py written
✅ requirements.txt written
✅ Dockerfile written
✅ README.md written

⏳ Creating Space: Wasim5647/MedAssist_Langchain ...
✅ Space created.
⏳ Adding HF_TOKEN secret...
✅ Secret added.
✅ Uploaded app.py
✅ Uploaded requirements.txt
✅ Uploaded Dockerfile
✅ Uploaded README.md

🚀 MedAssist deployed to HuggingFace Spaces!
🌐 URL: https://huggingface.co/spaces/Wasim5647/MedAssist_Langchain

⏳ Build takes 3-5 mins. First load takes ~5 mins
   while it downloads the dataset and builds the
   FAISS index. After that it responds instantly.

